In [3]:
# =====================================================================
# QUESTION 7 - SAVE FINAL OPTIMISED PIPELINE
# =====================================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression


SEED = 42

PATH = r"C:\Users\Sekomane\Desktop\Data ScienceTUT\NIWIS_Water Supply Reliability - Population_26-Sep-2026.csv"


# =====================================================================
# 1. LOAD DATA
# =====================================================================

df = pd.read_csv(PATH)

data = df.copy()

data = data.loc[
    :,
    ~data.columns.str.contains("^Unnamed")
]

data.columns = (
    data.columns
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
)


# =====================================================================
# 2. CLEAN NUMERIC DATA
# =====================================================================

def clean_number(series):

    return pd.to_numeric(
        series.astype(str)
              .str.replace(" ", "", regex=False)
              .str.replace(",", "", regex=False)
              .str.strip(),
        errors="coerce"
    )


for col in data.columns:

    if "%" not in col and col != "WSA Name":

        data[col] = clean_number(
            data[col]
        )


for col in data.columns:

    if "%" in col:

        data[col] = pd.to_numeric(
            data[col]
                .astype(str)
                .str.replace("%", "", regex=False)
                .str.strip(),
            errors="coerce"
        )


# =====================================================================
# 3. TARGET
# =====================================================================

data["low_reliability"] = (
    data["% Population Reliable Supply"] < 60
).astype(int)


# =====================================================================
# 4. REMOVE LEAKAGE
# =====================================================================

leakage_columns = [

    "Population Reliable Supply",
    "% Population Reliable Supply",

    "Urban Population Reliable Supply",
    "% Urban Population Reliable Supply",

    "Rural Population Reliable Supply",
    "% Rural Population Reliable Supply"
]


y = data["low_reliability"]


X = data.drop(
    columns=[
        "low_reliability",
        "WSA Name"
    ] + leakage_columns,
    errors="ignore"
).copy()


# =====================================================================
# 5. FEATURE ENGINEERING
# =====================================================================

X["population_access_rate"] = (
    X["Population with access to water"]
    / X["Population"]
) * 100


X["urban_access_rate"] = (
    X["Urban Population with access to water"]
    / X["Urban Population"]
) * 100


X["rural_access_rate"] = (
    X["Rural Population with access to water"]
    / X["Rural Population"]
) * 100


X["urban_population_share"] = (
    X["Urban Population"]
    / X["Population"]
) * 100


X["rural_population_share"] = (
    X["Rural Population"]
    / X["Population"]
) * 100


X = X.replace(
    [np.inf, -np.inf],
    np.nan
)


# =====================================================================
# 6. SAME DEVELOPMENT / TEST SPLIT
# =====================================================================

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)


# =====================================================================
# 7. FINAL LOCKED PIPELINE
# =====================================================================

final_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "scaler",
        RobustScaler()
    ),

    (
        "classifier",
        LogisticRegression(
            C=1,
            penalty="l1",
            solver="liblinear",
            class_weight=None,
            max_iter=5000,
            random_state=SEED
        )
    )
])


# Train using development data
final_pipeline.fit(
    X_dev,
    y_dev
)


# =====================================================================
# 8. SAVE COMPLETE PIPELINE
# =====================================================================

MODEL_FILE = "final_water_reliability_pipeline.joblib"

joblib.dump(
    final_pipeline,
    MODEL_FILE
)


print("=" * 70)
print("FINAL MODEL SAVED SUCCESSFULLY")
print("=" * 70)

print("File:", MODEL_FILE)

print(
    "\nFeatures expected by the model:"
)

for feature in X.columns:

    print("-", feature)

ModuleNotFoundError: No module named 'pandas'